# 🫀 心臟衰竭生存風險預測 API 教學：實作 `/train` 線上重訓與超參數微調端點

> 對象：修習隨機森林分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 92~110 行的 `train_endpoint` 與 `train_save.py` 的 `train_and_save_model` 函數
>
> 這份 Notebook 會帶你**一步步拆解**隨機森林線上模型訓練、超參數調校（決策樹數量 `n_estimators` 與最大樹深 `max_depth`）與模型熱載入機制，並使用 `TestClient` 模擬完整 API 呼叫流程。

## 🎯 學習目標

- [ ] 理解隨機森林中決策樹數量 (`n_estimators`) 與最大深度 (`max_depth`) 對抗過擬合與泛化能力的影響
- [ ] 掌握 `train_and_save_model()` 如何訓練 100+ 棵決策樹並計算 11 項臨床特徵重要性 (Feature Importances)
- [ ] 看懂 FastAPI 訓練端點的 Pydantic 請求模型 (`TrainRequest`)
- [ ] 實現模型熱重載（Hot Reloading）邏輯，在線上更新記憶體中的 `model` 與 `feature_importances`
- [ ] 能使用 `TestClient` 發送 POST `/train` 請求，並驗證模型更新後的指標

## 📌 背景：`/train` 端點運作架構

```
[收到 TrainRequest: n_estimators, max_depth]
       │
       ▼
1. train_and_save_model(...)  ──> 重新擬合隨機森林，序列化至 heart_failure_model.joblib
       │
       ▼
2. 更新全域變數              ──> 更新 global model 與 feature_importances
       │
       ▼
3. 回傳結構化 JSON            ──> 回傳訓練/測試準確率、耗時與成功訊息
```

In [ ]:
# ============================================
# 0. 載入套件與環境設定
# ============================================
import os, sys
from pprint import pprint

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from train_save import train_and_save_model
print("✅ 成功匯入 train_and_save_model 模組！")

---
## Part 1：測試模型訓練與序列化儲存 (`train_and_save_model`)

先直接呼叫 `train_and_save_model` 觀察訓練耗時與 11 項臨床指標重要性排行。

In [ ]:
result = train_and_save_model(
    n_estimators=100,
    max_depth=4,
    test_size=0.25,
    random_state=42
)

print("\n--- 訓練回傳元數據摘要 ---")
print(f"訓練集準確率: {result['train_accuracy']:.4f}")
print(f"測試集準確率: {result['test_accuracy']:.4f}")
print(f"訓練耗時: {result['train_time']:.4f} 秒")
print("各項臨床特徵重要性 (Feature Importances):")
pprint(result['feature_importances'])

---
## Part 2：定義訓練請求的 Pydantic 模型

限制決策樹數量介於 10~500，最大深度限制在 1~20 之間防止記憶體爆炸。

In [ ]:
from pydantic import BaseModel, Field

class TrainRequest(BaseModel):
    n_estimators: int = Field(default=100, ge=10, le=500, description="決策樹數量 (10~500)")
    max_depth: int = Field(default=4, ge=1, le=20, description="樹的最大深度 (1~20)")

sample_req = TrainRequest(n_estimators=150, max_depth=5)
print("請求模型實例化成功：", sample_req.model_dump())

---
## Part 3：建立 FastAPI 應用與 `/train` 線上重訓端點

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Heart Failure Training API")

MODEL_PATH = "heart_failure_model.joblib"
GLOBAL_STATE = {}

@app.post("/train", summary="線上熱調參重訓模型")
def train_endpoint(req: TrainRequest):
    try:
        new_meta = train_and_save_model(
            n_estimators=req.n_estimators,
            max_depth=req.max_depth,
            model_path=MODEL_PATH
        )
        GLOBAL_STATE["model"] = new_meta["model"]
        GLOBAL_STATE["feature_importances"] = new_meta["feature_importances"]
        
        return {
            "status": "success",
            "message": f"心臟衰竭模型重訓完成！樹木量: {req.n_estimators}, 深度: {req.max_depth}",
            "train_accuracy": new_meta["train_accuracy"],
            "test_accuracy": new_meta["test_accuracy"],
            "timestamp": new_meta["timestamp"]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"訓練失敗: {str(e)}")

print("✅ FastAPI /train 端點註冊成功！")

---
## Part 4：使用 `TestClient` 模擬呼叫 `/train` 端點

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 發送測試請求 (將樹的數量調為 120，最大深度限制為 6)
payload = {
    "n_estimators": 120,
    "max_depth": 6
}

response = client.post("/train", json=payload)
print("HTTP 狀態碼：", response.status_code)
print("API 回傳內容：")
pprint(response.json())